In [20]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import numpy as np
import itertools


In [22]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [24]:
# ------------------------------
# Matrix Factorization Model
# ------------------------------
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, n_factors=32):
        super().__init__()
        self.user_factors = nn.Embedding(n_users, n_factors)
        self.item_factors = nn.Embedding(n_items, n_factors)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.tensor([0.0]))

        nn.init.normal_(self.user_factors.weight, std=0.1)
        nn.init.normal_(self.item_factors.weight, std=0.1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, users, items):
        dot = (self.user_factors(users) * self.item_factors(items)).sum(dim=1, keepdim=True)
        preds = dot + self.user_bias(users) + self.item_bias(items) + self.global_bias
        return preds.squeeze()


In [26]:
# ------------------------------
# Dataset Class
# ------------------------------
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user_id"].values, dtype=torch.long)
        self.items = torch.tensor(df["item_id"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]


In [28]:
# ------------------------------
# Training and Evaluation
# ------------------------------
def train_model(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for users, items, ratings in train_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        optimizer.zero_grad()
        preds = model(users, items)
        loss = criterion(preds, ratings)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(ratings)
    return np.sqrt(total_loss / len(train_loader.dataset))

def evaluate_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for users, items, ratings in loader:
            users, items, ratings = users.to(device), items.to(device), ratings.to(device)
            preds = model(users, items)
            loss = criterion(preds, ratings)
            total_loss += loss.item() * len(ratings)
    return np.sqrt(total_loss / len(loader.dataset))


In [30]:
df = pd.read_csv("dataset/u.data", sep="\t", names=["user_id", "item_id", "rating", "timestamp"])
df.drop(columns=["timestamp"], inplace=True)
df["user_id"] -= df["user_id"].min()
df["item_id"] -= df["item_id"].min()

n_users = df["user_id"].nunique()
n_items = df["item_id"].nunique()

train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.1, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.MSELoss()

param_grid = {
    "n_factors": [5,10, 16, 32, 64],
    "lr": [ 1e-3, 1e-2, 1e-1],
    "weight_decay": [0, 1e-6, 1e-5, 1e-4]
}

best_val_rmse = float("inf")
best_params = {}
best_model_state = None
total_comm_cost = 0
total_grad_shared = 0


In [38]:
for n_factors, lr, weight_decay in itertools.product(
        param_grid["n_factors"], param_grid["lr"], param_grid["weight_decay"]
    ):
        print(f"Testing n_factors={n_factors}, lr={lr}, weight_decay={weight_decay}")
        model = MatrixFactorization(n_users, n_items, n_factors).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        train_loader = DataLoader(RatingsDataset(train_df), batch_size=1024, shuffle=True)
        val_loader = DataLoader(RatingsDataset(val_df), batch_size=1024)

        comm_cost_epoch = 0
        grad_share_epoch = 0
        n_epoch = 30
        for epoch in range(n_epoch):  # fewer epochs for tuning speed
            train_rmse = train_model(model, train_loader, optimizer, criterion, device)

            if epoch == 0:
                param_bytes = sum(p.numel() for p in model.parameters()) * 4  # float32
                comm_cost_epoch = param_bytes
                grad_share_epoch = sum(p.numel() for p in model.parameters())

        total_comm_cost += comm_cost_epoch * n_epoch
        total_grad_shared += grad_share_epoch * n_epoch

        val_rmse = evaluate_model(model, val_loader, criterion, device)
        print(f"  => Val RMSE: {val_rmse:.4f}")

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_params = {
                "n_factors": n_factors,
                "lr": lr,
                "weight_decay": weight_decay
            }
            best_model_state = model.state_dict()

print("\nBest Parameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"✅ Best Validation RMSE: {best_val_rmse:.4f}")
print(f"📦 Total Communication Cost: {total_comm_cost / 1e6:.2f} MB")
print(f"🔁 Total Gradients Shared: {total_grad_shared}")

   

Testing n_factors=5, lr=0.001, weight_decay=0
  => Val RMSE: 0.9531
Testing n_factors=5, lr=0.001, weight_decay=1e-06
  => Val RMSE: 0.9503
Testing n_factors=5, lr=0.001, weight_decay=1e-05
  => Val RMSE: 0.9529
Testing n_factors=5, lr=0.001, weight_decay=0.0001
  => Val RMSE: 0.9619
Testing n_factors=5, lr=0.01, weight_decay=0
  => Val RMSE: 0.9807
Testing n_factors=5, lr=0.01, weight_decay=1e-06
  => Val RMSE: 0.9648
Testing n_factors=5, lr=0.01, weight_decay=1e-05
  => Val RMSE: 0.9554
Testing n_factors=5, lr=0.01, weight_decay=0.0001
  => Val RMSE: 0.9121
Testing n_factors=5, lr=0.1, weight_decay=0
  => Val RMSE: 1.0273
Testing n_factors=5, lr=0.1, weight_decay=1e-06
  => Val RMSE: 1.0118
Testing n_factors=5, lr=0.1, weight_decay=1e-05
  => Val RMSE: 1.0123
Testing n_factors=5, lr=0.1, weight_decay=0.0001
  => Val RMSE: 0.9734
Testing n_factors=10, lr=0.001, weight_decay=0
  => Val RMSE: 0.9388
Testing n_factors=10, lr=0.001, weight_decay=1e-06
  => Val RMSE: 0.9450
Testing n_facto

In [44]:
sum(p.numel() for p in model.parameters())

170626

In [54]:
for p in model.parameters():
    print(p)

Parameter containing:
tensor([3.2149], requires_grad=True)
Parameter containing:
tensor([[ 0.3582, -0.2232, -0.1180,  ...,  0.3060,  0.1586, -0.6018],
        [ 0.1944,  0.0571, -0.0944,  ..., -0.0065,  0.0408, -0.0976],
        [ 0.3961,  0.5575,  0.1927,  ..., -0.1754, -0.2915, -0.1035],
        ...,
        [-0.0717, -0.2150,  0.0213,  ..., -0.0688,  0.0129,  0.0437],
        [ 0.0527, -0.2243,  0.0886,  ...,  0.1272, -0.0666,  0.0942],
        [-0.2032,  0.3294,  0.5364,  ...,  0.0277,  0.1902, -0.2118]],
       requires_grad=True)
Parameter containing:
tensor([[ 8.1454e-02,  2.3640e-02,  2.3625e-02,  ..., -5.2877e-01,
         -5.8336e-01, -4.2034e-02],
        [ 1.6810e-03, -1.8357e-01,  4.0740e-01,  ..., -1.1609e-01,
          2.3084e-01, -3.6304e-01],
        [ 2.0784e-01, -2.4865e-01,  1.0649e-01,  ...,  1.0246e-02,
          3.4141e-01, -3.6083e-02],
        ...,
        [ 6.3390e-02,  1.0523e-02, -2.5004e-02,  ...,  3.1849e-02,
          3.8207e-02, -2.1089e-02],
        [-6

In [42]:
# Final test evaluation
test_loader = DataLoader(RatingsDataset(test_df), batch_size=1024)
final_model = MatrixFactorization(n_users, n_items, best_params["n_factors"]).to(device)
final_model.load_state_dict(best_model_state)
test_rmse = evaluate_model(final_model, test_loader, criterion, device)
print(f"✅ Test RMSE with best model: {test_rmse:.4f}")


✅ Test RMSE with best model: 0.9211
